In [1]:
"""
Retention Decision System

Goal:
- Combine churn prediction and A/B testing
- Output actionable business decisions
"""

import pandas as pd
import joblib

In [2]:
"""
Load trained model and processed dataset.
"""

# Load model (we will save it shortly if not done)
model = joblib.load("../models/churn_model.pkl")

df = pd.read_csv("../data/processed/churn_features.csv")

df.head()

,creditscore,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,geography_Germany,geography_Spain,gender_Male,balance_per_product,credit_per_age,tenure_per_age
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False,0.00,14.395349,0.046512
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False,41903.93,14.476190,0.023810
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False,39915.20,11.674419,0.186047
3,699,39,1,0.00,2,0,0,93826.63,0,False,False,False,0.00,17.475000,0.025000
4,850,43,2,125510.82,1,1,1,79084.10,0,False,True,False,62755.41,19.318182,0.045455


In [3]:
"""
Generate churn probabilities.
"""

X = df.drop("exited", axis=1)

df["churn_probability"] = model.predict_proba(X)[:, 1]

df.head()

,creditscore,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,geography_Germany,geography_Spain,gender_Male,balance_per_product,credit_per_age,tenure_per_age,churn_probability
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False,0.00,14.395349,0.046512,0.521640
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False,41903.93,14.476190,0.023810,0.471397
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False,39915.20,11.674419,0.186047,0.867887
3,699,39,1,0.00,2,0,0,93826.63,0,False,False,False,0.00,17.475000,0.025000,0.243415
4,850,43,2,125510.82,1,1,1,79084.10,0,False,True,False,62755.41,19.318182,0.045455,0.338785


In [4]:
"""
Categorize customers into risk groups.
"""

def assign_risk(prob):
    if prob >= 0.6:
        return "High"
    elif prob >= 0.4:
        return "Medium"
    else:
        return "Low"

df["risk_level"] = df["churn_probability"].apply(assign_risk)

df["risk_level"].value_counts()

risk_level
Low       6407
High      1799
Medium    1794
Name: count, dtype: int64

In [5]:
"""
Assign best strategy based on A/B results.

We determined:
Strategy A is better
"""

def assign_strategy(risk):
    if risk == "High":
        return "Strategy A"
    else:
        return "No Action"

df["recommended_action"] = df["risk_level"].apply(assign_strategy)

df.head()

,creditscore,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,geography_Germany,geography_Spain,gender_Male,balance_per_product,credit_per_age,tenure_per_age,churn_probability,risk_level,recommended_action
0,619,42,2,0.00,1,1,1,101348.88,1,False,False,False,0.00,14.395349,0.046512,0.521640,Medium,No Action
1,608,41,1,83807.86,1,0,1,112542.58,0,False,True,False,41903.93,14.476190,0.023810,0.471397,Medium,No Action
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False,39915.20,11.674419,0.186047,0.867887,High,Strategy A
3,699,39,1,0.00,2,0,0,93826.63,0,False,False,False,0.00,17.475000,0.025000,0.243415,Low,No Action
4,850,43,2,125510.82,1,1,1,79084.10,0,False,True,False,62755.41,19.318182,0.045455,0.338785,Low,No Action


In [6]:
"""
Inspect high-risk customers.
"""

high_risk_df = df[df["risk_level"] == "High"]

print("Number of high-risk customers:", len(high_risk_df))

high_risk_df.head()

Number of high-risk customers: 1799


,creditscore,age,tenure,balance,numofproducts,hascrcard,isactivemember,estimatedsalary,exited,geography_Germany,geography_Spain,gender_Male,balance_per_product,credit_per_age,tenure_per_age,churn_probability,risk_level,recommended_action
2,502,42,8,159660.80,3,1,0,113931.57,1,False,False,False,39915.200,11.674419,0.186047,0.867887,High,Strategy A
7,376,29,4,115046.74,4,1,0,119346.88,1,True,False,False,23009.348,12.533333,0.133333,0.891277,High,Strategy A
16,653,58,1,132602.88,1,1,0,5097.67,1,True,False,True,66301.440,11.067797,0.016949,0.925041,High,Strategy A
18,587,45,6,0.00,1,0,0,158684.81,0,False,True,True,0.000,12.760870,0.130435,0.678695,High,Strategy A
28,574,43,3,141349.43,1,1,1,100187.43,0,True,False,False,70674.715,13.045455,0.068182,0.711955,High,Strategy A


In [7]:
"""
Summarize decisions.
"""

summary = df.groupby("risk_level")["recommended_action"].count()

print(summary)

risk_level
High      1799
Low       6407
Medium    1794
Name: recommended_action, dtype: int64


In [8]:
"""
Final system interpretation.
"""

print("=" * 60)
print("RETENTION DECISION SYSTEM")
print("=" * 60)

print("\n1. Churn Model:")
print("- Predicts probability of customer churn")

print("\n2. Risk Segmentation:")
print("- High risk: targeted for intervention")
print("- Medium/Low risk: monitored")

print("\n3. A/B Testing Insight:")
print("- Strategy A performs best (higher conversion)")

print("\n4. Final Recommendation:")
print("- Apply Strategy A to high-risk customers")

print("=" * 60)

RETENTION DECISION SYSTEM

1. Churn Model:
- Predicts probability of customer churn

2. Risk Segmentation:
- High risk: targeted for intervention
- Medium/Low risk: monitored

3. A/B Testing Insight:
- Strategy A performs best (higher conversion)

4. Final Recommendation:
- Apply Strategy A to high-risk customers


## Final System Overview

This project combines predictive modeling and experimentation to support business decision-making.

1. A churn model is used to identify high-risk customers
2. A/B testing is used to determine the most effective intervention
3. High-risk customers are targeted with the best-performing strategy

### Key Insight

Rather than only predicting churn, this system enables actionable retention strategies.

### Limitation

The A/B testing dataset is not directly linked to churn. Conversion is used as a proxy for engagement, which is assumed to influence retention.